In [ ]:
# install unsloth 
# !pip install unsloth langchain_anthropic langchain_core

In [ ]:
import os
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from transformers import TrainingArguments, DataCollatorForSeq2Seq
import os
from unsloth.chat_templates import get_chat_template

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

In [ ]:
max_seq_length = 5000
dtype = None  
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1", # Changed from phi-3 to llama-3.2
    mapping = {
        "role": "role",
        "content": "content",
        "user": "user",
        "assistant": "assistant",
        "system": "system" 
    }
)

In [ ]:
# Data preparation function
def formatting_prompts_func(examples):
    messages_list = examples["messages"]
    texts = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False) 
            for messages in messages_list]
    return {"text": texts}


# Load and process datasets
train_file = "data/input/training_data.jsonl"
eval_file = "data/input/validation_data.jsonl"
train_dataset = load_dataset('json', data_files=train_file, split='train')
print(f"Training dataset size: {len(train_dataset)}")
eval_dataset = load_dataset('json', data_files=eval_file, split='train')
print(f"Eval dataset size: {len(eval_dataset)}")

# train_dataset = train_dataset.select(range(10))
# eval_dataset = eval_dataset.select(range(1))

# Format the datasets
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# Define LoRA parameter sets
lora_params = {
    # 'tiny': {
    #     'r': 8,
    #     'lora_alpha': 128,
    #     'lora_dropout': 0.2,
    #     'use_rslora': False
    # },
    'small': {
        'r': 32,
        'lora_alpha': 256,
        'lora_dropout': 0.2,
        'use_rslora': False
    },
    # 'medium': {
    #     'r': 64,
    #     'lora_alpha': 384,
    #     'lora_dropout': 0.2,
    #     'use_rslora': False
    # },
    # 'large': {
    #     'r': 128,
    #     'lora_alpha': 512,
    #     'lora_dropout': 0.2,
    #     'use_rslora': False
    # },
    # 'xlarge': {
    #     'r': 128,
    #     'lora_alpha': 512,
    #     'lora_dropout': 0.2,
    #     'use_rslora': True  # Testing rsLoRA with largest config
    # }
}

# Define training parameter sets
training_params = {
    # 'very_conservative': {
    #     'learning_rate': 1e-5,
    #     'num_train_epochs': 2,
    #     'weight_decay': 0.01,
    #     'max_grad_norm': 0.3,
    #     'eval_steps': 5
    # },
    # 'conservative': {
    #     'learning_rate': 2e-5,
    #     'num_train_epochs': 4,
    #     'weight_decay': 0.02,
    #     'max_grad_norm': 0.4,
    #     'eval_steps': 10
    # },
    'moderate': {
        'learning_rate': 5e-5,
        'num_train_epochs': 6,
        'weight_decay': 0.03,
        'max_grad_norm': 0.5,
        'eval_steps': 15
    },
    'aggressive': {
        'learning_rate': 1e-4,
        'num_train_epochs': 8,
        'weight_decay': 0.04,
        'max_grad_norm': 0.6,
        'eval_steps': 20
    },
    'very_aggressive': {
        'learning_rate': 2e-4,
        'num_train_epochs': 10,
        'weight_decay': 0.05,
        'max_grad_norm': 0.7,
        'eval_steps': 25
    }
}

In [ ]:
def run_training_combination(base_model, tokenizer, train_dataset, eval_dataset, 
                           lora_config, train_config, config_name, max_seq_length=5000):
    """Run training for a specific combination of LoRA and training parameters"""
    
    # Set up model with LoRA configuration
    model = FastLanguageModel.get_peft_model(
        base_model,
        r=lora_config['r'],
        lora_alpha=lora_config['lora_alpha'],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj", "lm_head"],
        lora_dropout=lora_config['lora_dropout'],
        bias="none",
        use_gradient_checkpointing="unsloth",
        use_rslora=lora_config['use_rslora'],
        loftq_config=None
    )
    
    # Set up training arguments
    training_args = TrainingArguments(
        per_device_train_batch_size=10,
        per_device_eval_batch_size=10,
        gradient_accumulation_steps=20,
        warmup_steps=50,
        num_train_epochs=train_config['num_train_epochs'],
        learning_rate=train_config['learning_rate'],
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=train_config['eval_steps'],
        eval_steps=train_config['eval_steps'],
        eval_strategy="steps",
        optim="adamw_8bit",
        weight_decay=train_config['weight_decay'],
        lr_scheduler_type="cosine",
        max_grad_norm=train_config['max_grad_norm'],
        output_dir=f"models_{config_name}"
    )
    
    # Initialize trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        dataset_num_proc=2,
        packing=False,
        args=training_args
    )
    
    # Apply response-only training
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
        response_part="<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    
    # Train model
    trainer_stats = trainer.train()
    
    # Save model and tokenizer
    save_dir = f"{config_name}"
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    
    return trainer_stats

# Run grid search
def run_grid_search(base_model, tokenizer, train_dataset, eval_dataset):
    results = {}
    
    for lora_name, lora_config in lora_params.items():
        for train_name, train_config in training_params.items():
            config_name = f"{lora_name}_{train_name}"
            print(f"\nTraining configuration: {config_name}")
            
            try:
                stats = run_training_combination(
                    base_model=base_model,
                    tokenizer=tokenizer,
                    train_dataset=train_dataset,
                    eval_dataset=eval_dataset,
                    lora_config=lora_config,
                    train_config=train_config,
                    config_name=config_name
                )
                results[config_name] = stats
                
                # Clean up
                torch.cuda.empty_cache()
                
            except Exception as e:
                print(f"Error training {config_name}: {str(e)}")
                continue
    
    return results

results = run_grid_search(model, tokenizer, train_dataset, eval_dataset)